In [1]:
"""
After manual analysis, I've found some mistakes happening in the DB results.
In this notebook, I pick the incorrect records to test the DB_1record_evaluator, seeing if it successfully tells that these records are incorrect.
Extraction of text_doc and file_path_doc is also done here, so that it can be done once for 1 notegroup instead of repeatedly done for each record
"""
from oral_notes.s1_extract.doc_loader import GoogleDriveLoader
from oral_notes.s1_extract.text_extractor import TextExtractor
from oral_notes.evaluate.DB_1record_evaluator import DB1recordEvaluator
import sqlite3

pipeline_type="baseline_v3"
DB_PATH = "DB/oedb_baseline_v3.db"
schema_path = "data/metadata_DB/schema_v3.yaml"
prompt_path_evaluator = "data/prompt_templates/prompt_evaluator.yaml"
service_account_file="config/service_account_key.json"

def textdoc_extractor(notegroup_id, DB_PATH, service_account_file):
    with sqlite3.connect(DB_PATH) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT note_url_QA, note_url_PARTICIPANT
            FROM notegroups
            WHERE notegroupID = ?
        """, (notegroup_id,))
        row = cursor.fetchone()
        if row is None:
            raise ValueError(f"No notegroup found with ID {notegroup_id}")
        note_url_qa, note_url_participant = row

    file_loader = GoogleDriveLoader(service_account_file)
    extractor = TextExtractor()

    all_texts = {}
    all_drive_paths = {}
    for label, url in [("QA", note_url_qa), ("PARTICIPANT", note_url_participant)]:
        if not url:
            print(f"\n--- Skipping {label}: no URL ---")
            continue
        print(f"\n--- Loading and extracting text from {label} ---")
        result = file_loader.load(url)
        text = extractor.extract(result)
        all_texts[label] = f"[Data source: {result['name']}]\n{text}"
        all_drive_paths[label] = result['drive_path']
    combined_drive_paths = "|".join(all_drive_paths.values())
    combined_text = "\n---\n".join(
        t for t in [all_texts.get('PARTICIPANT', ''), all_texts.get('QA', '')] if t
    )
    return combined_text, combined_drive_paths

In [3]:
evaluator_version="initial_test_onv3"
notegroupID=1
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
# run the evaluator over the target records
records_to_evaluate = (
    [("participants", 6)]
    +[("questions", pk) for pk in range(1, 18)]   # 1-17
)

for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Rama - Note-taking 3 (1).docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Rama - Note-taking 3 (1).docx

--- Loading and extracting text from PARTICIPANT ---
Loading: Aanwezig sessie 1 (application/vnd.google-apps.document)


2026-06-22 16:49:28 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=1:
{
  "question_content": "How is your inburgering going so far?",
  "main_indicator": [
    "education",
    "onderwijs",
    "language",
    "taal"
  ]
}


Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Aanwezig sessie 1

--- Evaluating questions pk=1 ---


2026-06-22 16:49:29 | INFO     | utils.token_logger | Token usage [questions pk=1] attempt 1 — in: 7237 (cached: 6144), out: 26, cost: $0.002394
2026-06-22 16:49:29 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=1] correct=0 wrong_fields=['main_indicator']
2026-06-22 16:49:29 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=2:
{
  "question_content": "Where do you feel pressure in your life, with the inburgering? What feels difficult to handle?",
  "main_indicator": [
    "education",
    "onderwijs",
    "health",
    "zorg & welzijn"
  ]
}



--- Evaluating questions pk=2 ---


...

2026-06-22 16:49:43 | INFO     | utils.token_logger | Token usage [questions pk=15] attempt 1 — in: 7256 (cached: 6144), out: 23, cost: $0.002388
2026-06-22 16:49:43 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=15] correct=1 wrong_fields=[]
2026-06-22 16:49:43 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=16:
{
  "question_content": "What’s one thing that the municipality can do to make it easier for you to meet new people?",
  "main_indicator": [
    "bridges",
    "bruggen",
    "rights and responsibilities",
    "rechten & verantwoordelijkheden"
  ]
}



--- Evaluating questions pk=16 ---


2026-06-22 16:49:44 | INFO     | utils.token_logger | Token usage [questions pk=16] attempt 1 — in: 7252 (cached: 6144), out: 26, cost: $0.002413
2026-06-22 16:49:44 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=16] correct=0 wrong_fields=['main_indicator']
2026-06-22 16:49:44 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=17:
{
  "question_content": "From the survey, we have seen that many people want to improve their Dutch by practicing it with other people. In what ways would you like that to be set-up? What can the municipality do to facilitate this practice?",
  "main_indicator": [
    "language",
    "taal",
    "bridges",
    "bruggen"
  ]
}



--- Evaluating questions pk=17 ---


2026-06-22 16:49:46 | INFO     | utils.token_logger | Token usage [questions pk=17] attempt 1 — in: 7272 (cached: 6144), out: 23, cost: $0.002408
2026-06-22 16:49:46 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=17] correct=1 wrong_fields=[]


In [4]:
evaluator_version="initial_test_onv3"
notegroupID=2
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
# run the evaluator over the target records
records_to_evaluate = (
    [("questions", 20)]
    +[("participants", pk) for pk in range(17, 24)]
    +[("answers", pk) for pk in [138,145,152]]

)

for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Iyad - Note-taking 3.12 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Iyad - Note-taking 3.12

--- Loading and extracting text from PARTICIPANT ---
Loading: Aanwezig sessie 1 (application/vnd.google-apps.document)


2026-06-22 16:58:33 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=20:
{
  "question_content": "How is your inburgering going so far? Where do you feel pressure in your life, with the inburgering? What feels difficult to handle?",
  "main_indicator": [
    "education",
    "onderwijs"
  ]
}


Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Aanwezig sessie 1

--- Evaluating questions pk=20 ---


2026-06-22 16:58:34 | INFO     | utils.token_logger | Token usage [questions pk=20] attempt 1 — in: 5165 (cached: 0), out: 23, cost: $0.006686
2026-06-22 16:58:34 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=20] correct=1 wrong_fields=[]
2026-06-22 16:58:34 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=17:
{
  "session_identifier": "A",
  "gender": "Male",
  "learning_route": "Z route",
  "place_of_origin": "Syria",
  "municipality": "Gemert"
}



--- Evaluating participants pk=17 ---


...

2026-06-22 16:58:44 | INFO     | utils.token_logger | Token usage [answers pk=138] attempt 1 — in: 4811 (cached: 4096), out: 23, cost: $0.001636
2026-06-22 16:58:44 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=138] correct=1 wrong_fields=[]
2026-06-22 16:58:44 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=145:
{
  "questionID": 28,
  "participantID": 23,
  "answer_content_oriLAN": "Through voluntary work and participating in social and recreational activities.",
  "session_identifier": "E",
  "gender": "Female",
  "learning_route": "B1",
  "place_of_origin": "Syria",
  "question_content": "In what ways do you try to meet new people?"
}



--- Evaluating answers pk=145 ---


2026-06-22 16:58:45 | INFO     | utils.token_logger | Token usage [answers pk=145] attempt 1 — in: 4780 (cached: 4224), out: 23, cost: $0.001453
2026-06-22 16:58:45 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=145] correct=1 wrong_fields=[]
2026-06-22 16:58:45 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=152:
{
  "questionID": 29,
  "participantID": 23,
  "answer_content_oriLAN": "I am missing Dutch people with relevant professional experience and knowledge in architectural design. Also, language and integration content prevent me from fully connecting.",
  "session_identifier": "E",
  "gender": "Female",
  "learning_route": "B1",
  "place_of_origin": "Syria",
  "question_content": "When you look at your social connections now, who is missing? What kind of contacts and relationships do you want more of?"
}



--- Evaluating answers pk=152 ---


2026-06-22 16:58:46 | INFO     | utils.token_logger | Token usage [answers pk=152] attempt 1 — in: 4810 (cached: 4224), out: 23, cost: $0.001490
2026-06-22 16:58:46 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=152] correct=1 wrong_fields=[]


In [5]:
evaluator_version="initial_test_onv3"
notegroupID=3
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("participants", 27)]
    + [("answers", 189)]   # 28-31
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Note form Danna 22 Nov (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Notes/Note form Danna 22 Nov

--- Loading and extracting text from PARTICIPANT ---
Loading: Deelnemers lijst + indeling (application/vnd.google-apps.document)


2026-06-22 17:06:22 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=27:
{
  "full_name": "Lydia",
  "session_identifier": "L"
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Deelnemers lijst + indeling

--- Evaluating participants pk=27 ---


2026-06-22 17:06:23 | INFO     | utils.token_logger | Token usage [participants pk=27] attempt 1 — in: 7306 (cached: 0), out: 23, cost: $0.009362
2026-06-22 17:06:23 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=27] correct=1 wrong_fields=[]
2026-06-22 17:06:23 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=189:
{
  "questionID": 40,
  "participantID": 27,
  "answer_content_oriLAN": "they taught us, Dutch language. We took certificate. They help us with the Dutch; step by step. Beginner then more then more. We stopped. I did the exam and stopped. 3a2li battal yjammi3.\nFB and social media stuff are scam. Spam. They do it for different purposes. UWV dont help with work/\nThe most helpful is network. Gemeente.\nTelegram can help, but I dont know much about them. But they are serious.\nAwal ma tefta7 mawdou3 elshoghl mas2oul el COA be7ki ma dakhalni. Baas ta3it el baladdie bt3teenna nasa2i7 w kol eshie la 7ad m


--- Evaluating answers pk=189 ---


2026-06-22 17:06:25 | INFO     | utils.token_logger | Token usage [answers pk=189] attempt 1 — in: 7185 (cached: 6272), out: 31, cost: $0.002235
2026-06-22 17:06:25 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=189] correct=0 wrong_fields=['answer_content_oriLAN', 'participantID']


In [6]:
evaluator_version="initial_test_onv3"
notegroupID=4
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("answers", 250)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Fatih notes form 22 nov (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Notes/Fatih notes form 22 nov

--- Loading and extracting text from PARTICIPANT ---
Loading: Deelnemers lijst + indeling (application/vnd.google-apps.document)


2026-06-22 17:10:44 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=250:
{
  "questionID": 55,
  "participantID": 32,
  "answer_content_oriLAN": "We answered it before",
  "full_name": "Özcan ikiz",
  "session_identifier": "O.E",
  "language_group": [
    "Turks"
  ],
  "question_content": "If you currently have a job:\n● Where do you work at the moment?"
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Deelnemers lijst + indeling

--- Evaluating answers pk=250 ---


2026-06-22 17:10:45 | INFO     | utils.token_logger | Token usage [answers pk=250] attempt 1 — in: 15960 (cached: 0), out: 23, cost: $0.020180
2026-06-22 17:10:45 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=250] correct=1 wrong_fields=[]


In [7]:
evaluator_version="initial_test_onv3"
notegroupID=5
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("answers", 402)]
    + [("questions", pk) for pk in range(90, 108)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Naya notes (application/vnd.google-apps.document)


2026-06-22 17:13:11 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=402:
{
  "questionID": 106,
  "participantID": 35,
  "answer_content_oriLAN": "They embarrassed us in the work process, once they hear we are refugees, they kick us out. There is only one difference between refugees and others, and this shouldnt be taken into consideration, because i have the experience and the TWV.\nAdeel: Racist people here.",
  "full_name": "Arsalan",
  "session_identifier": "Arsalan",
  "participant_group": "Asylum seeker",
  "place_of_origin": "Pakistan",
  "language_group": [
    "English",
    "Dutch"
  ],
  "municipality": "Zaandam",
  "question_content": "When did the support not work well? What was missing in your opinion? What would you have preferred to see done differently?"
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Notes/Naya notes

--- Skipping PARTICIPANT: no URL ---

--- Evaluating answers pk=402 ---


2026-06-22 17:13:13 | INFO     | utils.token_logger | Token usage [answers pk=402] attempt 1 — in: 6699 (cached: 0), out: 23, cost: $0.008604
2026-06-22 17:13:13 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=402] correct=1 wrong_fields=[]
2026-06-22 17:13:13 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=90:
{
  "question_content": "Specific to the AZC context:\n● Did the municipal contact person help you find work? If so, how? Provide concrete examples from your own experience. If not, why not? Provide concrete examples from your own experience.",
  "main_indicator": [
    "work",
    "werk & inkomen",
    "huisvesting & opvang"
  ]
}



--- Evaluating questions pk=90 ---


...

2026-06-22 17:13:31 | INFO     | utils.token_logger | Token usage [questions pk=105] attempt 1 — in: 7007 (cached: 5888), out: 23, cost: $0.002365
2026-06-22 17:13:31 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=105] correct=1 wrong_fields=[]
2026-06-22 17:13:31 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=106:
{
  "question_content": "When did the support not work well? What was missing in your opinion? What would you have preferred to see done differently?",
  "main_indicator": [
    "work",
    "werk & inkomen"
  ]
}



--- Evaluating questions pk=106 ---


2026-06-22 17:13:32 | INFO     | utils.token_logger | Token usage [questions pk=106] attempt 1 — in: 6994 (cached: 5888), out: 23, cost: $0.002349
2026-06-22 17:13:32 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=106] correct=1 wrong_fields=[]
2026-06-22 17:13:32 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=107:
{
  "question_content": "What is your advice to a person searching for work during their stay in the AZC? Give a tip based on your own experience!",
  "main_indicator": [
    "work",
    "werk & inkomen",
    "huisvesting & opvang"
  ]
}



--- Evaluating questions pk=107 ---


2026-06-22 17:13:33 | INFO     | utils.token_logger | Token usage [questions pk=107] attempt 1 — in: 7002 (cached: 5888), out: 23, cost: $0.002359
2026-06-22 17:13:33 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=107] correct=1 wrong_fields=[]


In [8]:
evaluator_version="initial_test_onv3"
notegroupID=6
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("participants", 36)]
    + [("answers", pk) for pk in [424,432]]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Copy of Ale_ Note-taking form 28.11 (English translation) (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Copy of Ale_ Note-taking form 28.11 (English translation)

--- Loading and extracting text from PARTICIPANT ---
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)


2026-06-22 17:21:00 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=36:
{
  "full_name": "Francielis Rivas",
  "session_identifier": "Francielis",
  "learning_route": "B1-route",
  "place_of_origin": "Venezuela",
  "language_group": [
    "Spaans"
  ],
  "first_arrival_date": "2021-01-01",
  "municipality": "Amsterdam"
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2

--- Evaluating participants pk=36 ---


2026-06-22 17:21:02 | INFO     | utils.token_logger | Token usage [participants pk=36] attempt 1 — in: 7779 (cached: 0), out: 34, cost: $0.010064
2026-06-22 17:21:02 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=36] correct=0 wrong_fields=['first_arrival_date', 'municipality', 'other_information']
2026-06-22 17:21:02 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=424:
{
  "questionID": 127,
  "participantID": 36,
  "answer_content_oriLAN": "Yes, because it’s not easy to manage study time, work, and integration. Integration is not only the courses but also the language, which is like studying a full university degree. It requires full-time attention.",
  "full_name": "Francielis Rivas",
  "session_identifier": "Francielis",
  "learning_route": "B1-route",
  "place_of_origin": "Venezuela",
  "language_group": [
    "Spaans"
  ],
  "first_arrival_date": "2021-01-01",
  "municipality": "Amsterdam",
  "question_c


--- Evaluating answers pk=424 ---


2026-06-22 17:21:04 | INFO     | utils.token_logger | Token usage [answers pk=424] attempt 1 — in: 7516 (cached: 6656), out: 23, cost: $0.002137
2026-06-22 17:21:04 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=424] correct=1 wrong_fields=[]
2026-06-22 17:21:04 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=432:
{
  "questionID": 135,
  "participantID": 36,
  "answer_content_oriLAN": "What is needed to improve this? (examples: motivation, resilience, networks, trust in the person or organisation, communication style, etc.)",
  "full_name": "Francielis Rivas",
  "session_identifier": "Francielis",
  "learning_route": "B1-route",
  "place_of_origin": "Venezuela",
  "language_group": [
    "Spaans"
  ],
  "first_arrival_date": "2021-01-01",
  "municipality": "Amsterdam",
  "question_content": "What is needed to improve this? (examples: motivation, resilience, networks, trust in the person or organisation, communicat


--- Evaluating answers pk=432 ---


2026-06-22 17:21:05 | INFO     | utils.token_logger | Token usage [answers pk=432] attempt 1 — in: 7503 (cached: 6912), out: 28, cost: $0.001883
2026-06-22 17:21:05 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=432] correct=0 wrong_fields=['answer_content_oriLAN']


In [9]:
evaluator_version="initial_test_onv3"
notegroupID=7
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("questions", 140)]
    +[("participants", pk) for pk in range(37, 41)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Danna: Note-taking form 28.11 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Danna: Note-taking form 28.11

--- Loading and extracting text from PARTICIPANT ---
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)


2026-06-22 17:27:00 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=140:
{
  "question_content": "Reflection on “Pathways to Work”:\nWhich of these tools have you used to find a job? Which have you not used? Why or why not?\n️ With help from the municipality or my case manager\n🧾 Through an employment agency or job coach\n‍ Through school, language class, or education\n🫶 With support from VluchtelingenWerk, UAF, NewBees, or another organisation\n🤝 I contact employers directly\n👐 Through volunteer work or an internship\n👪 Through my social network, friends, or family\n💬 With help from a buddy, mentor, or volunteer\n💻 Through online groups (e.g., Facebook, WhatsApp, Telegram, other)\n📱 Through social media (Facebook, LinkedIn, other)\n🏫 Through the UWV\n🌾 I use job-vacancy websites (Indeed, Werk.nl, RefugeeWork, NationaleVacaturebank)"
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2

--- Evaluating questions pk=140 ---


2026-06-22 17:27:02 | INFO     | utils.token_logger | Token usage [questions pk=140] attempt 1 — in: 8861 (cached: 0), out: 23, cost: $0.011306
2026-06-22 17:27:02 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=140] correct=1 wrong_fields=[]
2026-06-22 17:27:02 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=37:
{
  "full_name": "Rawa Alshumry",
  "session_identifier": "R",
  "language_group": [
    "Arabic"
  ],
  "municipality": "Rotterdam"
}



--- Evaluating participants pk=37 ---


...

2026-06-22 17:27:05 | INFO     | utils.token_logger | Token usage [participants pk=38] attempt 1 — in: 8636 (cached: 7552), out: 26, cost: $0.002559
2026-06-22 17:27:05 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=38] correct=0 wrong_fields=['municipality']
2026-06-22 17:27:05 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=39:
{
  "full_name": "Abdullah Najjar",
  "session_identifier": "A.N",
  "place_of_origin": "Syria",
  "language_group": [
    "Arabic",
    "English",
    "Turkish"
  ],
  "municipality": "Utrecht"
}



--- Evaluating participants pk=39 ---


2026-06-22 17:27:06 | INFO     | utils.token_logger | Token usage [participants pk=39] attempt 1 — in: 8656 (cached: 7552), out: 33, cost: $0.002654
2026-06-22 17:27:06 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=39] correct=0 wrong_fields=['place_of_origin', 'language_group', 'municipality']
2026-06-22 17:27:06 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=40:
{
  "full_name": "Victor",
  "session_identifier": "V",
  "place_of_origin": "Syria",
  "first_arrival_date": "2019-11-11",
  "municipality": "Rotterdam",
  "age": 41
}



--- Evaluating participants pk=40 ---


2026-06-22 17:27:07 | INFO     | utils.token_logger | Token usage [participants pk=40] attempt 1 — in: 8657 (cached: 7552), out: 34, cost: $0.002665
2026-06-22 17:27:07 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=40] correct=0 wrong_fields=['place_of_origin', 'first_arrival_date', 'age']


In [10]:
evaluator_version="initial_test_onv3"
notegroupID=8
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("participants", pk) for pk in range(41, 45)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Fatih: Note-taking form 28.11 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Fatih: Note-taking form 28.11

--- Loading and extracting text from PARTICIPANT ---
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)


2026-06-22 17:34:00 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=41:
{
  "full_name": "Nuri Berber",
  "session_identifier": "Nuri",
  "language_group": [
    "Turks"
  ],
  "municipality": "Raalte"
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2

--- Evaluating participants pk=41 ---


2026-06-22 17:34:03 | INFO     | utils.token_logger | Token usage [participants pk=41] attempt 1 — in: 14980 (cached: 0), out: 29, cost: $0.019015
2026-06-22 17:34:03 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=41] correct=0 wrong_fields=['session_identifier', 'municipality']
2026-06-22 17:34:03 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=42:
{
  "full_name": "Kamile özbek",
  "session_identifier": "Kamile",
  "place_of_origin": "Turkije",
  "language_group": [
    "Turks"
  ]
}



--- Evaluating participants pk=42 ---


2026-06-22 17:34:04 | INFO     | utils.token_logger | Token usage [participants pk=42] attempt 1 — in: 14982 (cached: 13952), out: 23, cost: $0.003262
2026-06-22 17:34:04 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=42] correct=1 wrong_fields=[]
2026-06-22 17:34:04 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=43:
{
  "full_name": "Kemal OZDEN",
  "session_identifier": "Kemal",
  "place_of_origin": "Turkije",
  "language_group": [
    "Turks"
  ],
  "municipality": "Raalte",
  "age": 50
}



--- Evaluating participants pk=43 ---


2026-06-22 17:34:07 | INFO     | utils.token_logger | Token usage [participants pk=43] attempt 1 — in: 14996 (cached: 13952), out: 28, cost: $0.003329
2026-06-22 17:34:07 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=43] correct=0 wrong_fields=['municipality', 'age']
2026-06-22 17:34:07 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=44:
{
  "full_name": "Fatih Dogandemir",
  "session_identifier": "Fatih",
  "participant_group": "Permit holder",
  "place_of_origin": "Turkije",
  "language_group": [
    "Turks"
  ]
}



--- Evaluating participants pk=44 ---


2026-06-22 17:34:08 | INFO     | utils.token_logger | Token usage [participants pk=44] attempt 1 — in: 14992 (cached: 13952), out: 33, cost: $0.003374
2026-06-22 17:34:08 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=44] correct=0 wrong_fields=['participant_group', 'place_of_origin', 'language_group']


In [11]:
evaluator_version="initial_test_onv3"
notegroupID=9
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("answers", pk) for pk in range(614, 622)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Nesrine: Note-taking form 28.11 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Nesrine: Note-taking form 28.11

--- Loading and extracting text from PARTICIPANT ---
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)


2026-06-22 17:39:42 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=614:
{
  "questionID": 203,
  "participantID": 45,
  "full_name": "Ali Banat",
  "session_identifier": "Al",
  "place_of_origin": "Syria",
  "language_group": [
    "Arabic"
  ],
  "municipality": "Den Haag",
  "question_content": "If yes: Do you still use the same method you used when finding your first job? (Which one?)"
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2

--- Evaluating answers pk=614 ---


2026-06-22 17:39:44 | INFO     | utils.token_logger | Token usage [answers pk=614] attempt 1 — in: 9270 (cached: 0), out: 31, cost: $0.011898
2026-06-22 17:39:44 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=614] correct=0 wrong_fields=['answer_content_oriLAN', 'participantID']
2026-06-22 17:39:44 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=615:
{
  "questionID": 203,
  "participantID": 46,
  "full_name": "Abdelkarim Alahmad",
  "session_identifier": "Ab",
  "learning_route": "B1-route",
  "place_of_origin": "Syria",
  "language_group": [
    "Arabic"
  ],
  "question_content": "If yes: Do you still use the same method you used when finding your first job? (Which one?)"
}



--- Evaluating answers pk=615 ---


...

2026-06-22 17:39:51 | INFO     | utils.token_logger | Token usage [answers pk=619] attempt 1 — in: 9271 (cached: 8704), out: 31, cost: $0.002107
2026-06-22 17:39:51 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=619] correct=0 wrong_fields=['answer_content_oriLAN', 'participantID']
2026-06-22 17:39:51 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=620:
{
  "questionID": 204,
  "participantID": 47,
  "full_name": "Waleed omar Bin mahram",
  "session_identifier": "W",
  "language_group": [
    "Arabic"
  ],
  "municipality": "Venlo",
  "question_content": "If yes: Or are you using a different method now? If so, which one?"
}



--- Evaluating answers pk=620 ---


2026-06-22 17:39:52 | INFO     | utils.token_logger | Token usage [answers pk=620] attempt 1 — in: 9263 (cached: 8704), out: 28, cost: $0.002067
2026-06-22 17:39:52 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=620] correct=0 wrong_fields=['answer_content_oriLAN']
2026-06-22 17:39:52 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=621:
{
  "questionID": 204,
  "participantID": 48,
  "full_name": "Latifa Al Ajeel",
  "session_identifier": "L",
  "language_group": [
    "Arabic"
  ],
  "municipality": "Groningen",
  "question_content": "If yes: Or are you using a different method now? If so, which one?"
}



--- Evaluating answers pk=621 ---


2026-06-22 17:39:53 | INFO     | utils.token_logger | Token usage [answers pk=621] attempt 1 — in: 9261 (cached: 8704), out: 28, cost: $0.002064
2026-06-22 17:39:53 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=621] correct=0 wrong_fields=['answer_content_oriLAN']


In [12]:
evaluator_version="initial_test_onv3"
notegroupID=10
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("answers", 706)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Reza: Note-taking form 29.11 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Reza: Note-taking form 29.11

--- Loading and extracting text from PARTICIPANT ---
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)


2026-06-22 17:43:38 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=706:
{
  "questionID": 239,
  "participantID": 50,
  "answer_content_oriLAN": "آیدا: دقیق نمی‌دانم. شاید چون من تازه آمده ام به این موضوع بر نخوردم.",
  "answer_content_EN": "Aida: I’m not sure. Maybe because I only recently started the integration course, I haven’t experienced this situation yet.",
  "full_name": "Aida",
  "session_identifier": "Aida",
  "language_group": [
    "Farsi"
  ],
  "question_content": "If yes, how? Give concrete examples from your own experience.\nIf not, why not? Give concrete examples from your own experience."
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2

--- Evaluating answers pk=706 ---


2026-06-22 17:43:41 | INFO     | utils.token_logger | Token usage [answers pk=706] attempt 1 — in: 9518 (cached: 0), out: 23, cost: $0.012127
2026-06-22 17:43:41 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=706] correct=1 wrong_fields=[]


In [13]:
evaluator_version="initial_test_onv3"
notegroupID=11
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("answers", 756)]
    +[("answers", pk) for pk in range(769, 773)]

)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Turkse_groep_verzamelde_data (application/vnd.google-apps.document)


2026-06-22 18:15:41 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=756:
{
  "questionID": 261,
  "participantID": 53,
  "answer_content_oriLAN": "\"Een mens heeft een eigen huis nodig. Leven in het kamp is erg moeilijk – en met kinderen is het nog veel zwaarder.\nS, Turkse, Asielzoeker, Vrouw:\"Het kamp was vroeger een gevangenis – dat merk je aan alles. Het imago is heel negatief, en daardoor kijkt de bevolking in E, Turkse, Asielzoeker, Vrouw:Den Helder met afstand naar ons.\"\n\"Ik mis mijn huis enorm. Leven in het kamp is zwaar. Kunt u alstublieft de fysieke omstandigheden en de hygiëne verbeteren?\"",
  "session_identifier": "S, Turkse, Asielzoeker, Vrouw",
  "gender": "Vrouw",
  "participant_group": "Asielzoeker",
  "place_of_origin": "Turkse",
  "language_group": [
    "Turks"
  ],
  "municipality": "Den Helder",
  "question_content": "Inspanningen\nOpvang"
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1341 - Expertpool Den Helder/2. Expertpool/expertpool notities notities/Notes session 1/Turkse_groep_verzamelde_data

--- Skipping PARTICIPANT: no URL ---

--- Evaluating answers pk=756 ---


2026-06-22 18:15:42 | INFO     | utils.token_logger | Token usage [answers pk=756] attempt 1 — in: 7963 (cached: 0), out: 28, cost: $0.010234
2026-06-22 18:15:42 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=756] correct=0 wrong_fields=['answer_content_oriLAN']
2026-06-22 18:15:42 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=769:
{
  "questionID": 266,
  "participantID": 54,
  "answer_content_oriLAN": "S, Turkse, Asielzoeker, Man  Ik wil door middel van taal leren werk vinden, mezelf kunnen uitdrukken en uit mijn isolement komen. Zelfs met COA kunnen we ons soms niet goed verstaanbaar maken. Taal is voor mij de sleutel tot zelfstandigheid, sociale contacten en mentale rust.( Indicator: Taalverwerving)\nE, Turkse, Asielzoeker, Vrouw Ik wil een eigen woning – omdat dat een basisrecht is. Met een huis kun je eindelijk echt beginnen aan integratie: je hebt buren, maakt contact, en wordt deel van de samenleving. Zon


--- Evaluating answers pk=769 ---


...

2026-06-22 18:15:45 | INFO     | utils.token_logger | Token usage [answers pk=770] attempt 1 — in: 8077 (cached: 7296), out: 31, cost: $0.002198
2026-06-22 18:15:45 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=770] correct=0 wrong_fields=['participantID', 'answer_content_oriLAN']
2026-06-22 18:15:45 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=771:
{
  "questionID": 268,
  "participantID": 54,
  "answer_content_oriLAN": "S, Turkse, Asielzoeker, Man  Wat goed gaat is dat ik gemotiveerd ben en zelfstandig leer. Ik onderneem veel, en ik zie ook kleine vooruitgangen. Maar ik ervaar veel obstakels: er is geen professionele begeleiding in het kamp, taallessen zijn beperkt, en ik heb geen toegang tot goede studiematerialen – boeken zijn te duur. Daarnaast merk ik dat mensen in Den Helder mij niet vertrouwen, omdat ze ons niet kennen. Het politieke klimaat is veranderd en dat voelen we elke dag. In de media verschijne


--- Evaluating answers pk=771 ---


2026-06-22 18:15:47 | INFO     | utils.token_logger | Token usage [answers pk=771] attempt 1 — in: 8196 (cached: 7296), out: 28, cost: $0.002317
2026-06-22 18:15:47 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=771] correct=0 wrong_fields=['answer_content_oriLAN']
2026-06-22 18:15:47 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=772:
{
  "questionID": 269,
  "participantID": 54,
  "answer_content_oriLAN": "S, Turkse, Asielzoeker, Man Van mezelf: blijven oefenen, blijven zoeken naar kansen, en mijn motivatie vasthouden. Van anderen: een netwerk dat mij serieus neemt, vertrouwen geeft, en taalcontact biedt. Van de gemeente: actieve samenwerking met COA, versnelde BSN- en verblijfsprocedures, en duidelijke informatie over de voorzieningen in de stad. Ook zou de gemeente gratis taalmaterialen kunnen aanbieden en helpen om het publieke beeld over ons te verbeteren. ( Indicator: Taalverwerving)\nE, Turkse, Asielzoeker


--- Evaluating answers pk=772 ---


2026-06-22 18:15:48 | INFO     | utils.token_logger | Token usage [answers pk=772] attempt 1 — in: 8189 (cached: 7296), out: 31, cost: $0.002338
2026-06-22 18:15:48 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=772] correct=0 wrong_fields=['answer_content_oriLAN', 'participantID']


In [14]:
evaluator_version="initial_test_onv3"
notegroupID=12
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("participants", pk) for pk in range(55, 58)]
    + [("questions", pk) for pk in range(270, 276)]
    + [("answers", pk) for pk in [824, 825]]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Data session 3 (AMV, Josja) (application/vnd.google-apps.document)


2026-06-22 18:22:31 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=55:
{
  "session_identifier": "Participant 1",
  "gender": "woman",
  "place_of_origin": "Iraq",
  "municipality": "Den Helder",
  "age": 19
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1341 - Expertpool Den Helder/2. Expertpool/expertpool notities notities/Notes session 1/Data session 3 (AMV, Josja)

--- Skipping PARTICIPANT: no URL ---

--- Evaluating participants pk=55 ---


2026-06-22 18:22:32 | INFO     | utils.token_logger | Token usage [participants pk=55] attempt 1 — in: 3633 (cached: 0), out: 23, cost: $0.004771
2026-06-22 18:22:32 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=55] correct=1 wrong_fields=[]
2026-06-22 18:22:32 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=56:
{
  "session_identifier": "Participant 2",
  "gender": "woman",
  "place_of_origin": "Somalia",
  "municipality": "Den Helder",
  "age": 17
}



--- Evaluating participants pk=56 ---


...

2026-06-22 18:22:40 | INFO     | utils.token_logger | Token usage [questions pk=275] attempt 1 — in: 3667 (cached: 3584), out: 26, cost: $0.000812
2026-06-22 18:22:40 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=275] correct=0 wrong_fields=['question_content']
2026-06-22 18:22:40 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=824:
{
  "questionID": 278,
  "participantID": 57,
  "answer_content_oriLAN": "The last adult wants to start working, but don’t know how.",
  "session_identifier": "Participant 3",
  "gender": "man",
  "place_of_origin": "Sudanese",
  "municipality": "Den Helder",
  "age": 19,
  "question_content": "Work and income"
}



--- Evaluating answers pk=824 ---


2026-06-22 18:22:41 | INFO     | utils.token_logger | Token usage [answers pk=824] attempt 1 — in: 3317 (cached: 2560), out: 23, cost: $0.001496
2026-06-22 18:22:41 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=824] correct=1 wrong_fields=[]
2026-06-22 18:22:41 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=825:
{
  "questionID": 279,
  "participantID": 57,
  "answer_content_oriLAN": "The ability to ask other people. He used to do it at COA, but not anymore. How doesn't feel taken seriously.",
  "session_identifier": "Participant 3",
  "gender": "man",
  "place_of_origin": "Sudanese",
  "municipality": "Den Helder",
  "age": 19,
  "question_content": "What do you need to find work?"
}



--- Evaluating answers pk=825 ---


2026-06-22 18:22:42 | INFO     | utils.token_logger | Token usage [answers pk=825] attempt 1 — in: 3334 (cached: 2688), out: 23, cost: $0.001373
2026-06-22 18:22:42 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=825] correct=1 wrong_fields=[]


In [15]:
evaluator_version="initial_test_onv3"
notegroupID=13
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("questions", pk) for pk in range(283, 291)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Data den Helder Ist session Ula (application/vnd.google-apps.document)


2026-06-22 18:28:31 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=283:
{
  "question_content": "Languages\nWhy did you choose this indicator?\nAchieve?\nObstacles?\nActions taken?\nWhat would help achieve this goal?",
  "main_indicator": [
    "language",
    "taal"
  ]
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1341 - Expertpool Den Helder/2. Expertpool/expertpool notities notities/Notes session 1/Data den Helder Ist session Ula

--- Skipping PARTICIPANT: no URL ---

--- Evaluating questions pk=283 ---


2026-06-22 18:28:32 | INFO     | utils.token_logger | Token usage [questions pk=283] attempt 1 — in: 3171 (cached: 0), out: 23, cost: $0.004194
2026-06-22 18:28:32 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=283] correct=1 wrong_fields=[]
2026-06-22 18:28:32 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=284:
{
  "question_content": "Housing\nWhy did you choose this indicator?\nAchieve?\nObstacles?\nActions taken?\nWhat would help achieve this goal?",
  "main_indicator": [
    "housing",
    "huisvesting & opvang"
  ]
}



--- Evaluating questions pk=284 ---


...

2026-06-22 18:28:37 | INFO     | utils.token_logger | Token usage [questions pk=288] attempt 1 — in: 3171 (cached: 2048), out: 23, cost: $0.001890
2026-06-22 18:28:37 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=288] correct=1 wrong_fields=[]
2026-06-22 18:28:37 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=289:
{
  "question_content": "Work\nWhat do you need to start working ?\nWhat are the obstacles?\nHow can the municipality help ?",
  "main_indicator": [
    "work",
    "werk & inkomen"
  ]
}



--- Evaluating questions pk=289 ---


2026-06-22 18:28:38 | INFO     | utils.token_logger | Token usage [questions pk=289] attempt 1 — in: 3167 (cached: 2048), out: 23, cost: $0.001885
2026-06-22 18:28:38 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=289] correct=1 wrong_fields=[]
2026-06-22 18:28:38 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=290:
{
  "question_content": "Collecting feedback\nHow would you like the municipality to collect feedback from you?"
}



--- Evaluating questions pk=290 ---


2026-06-22 18:28:39 | INFO     | utils.token_logger | Token usage [questions pk=290] attempt 1 — in: 3147 (cached: 2048), out: 23, cost: $0.001860
2026-06-22 18:28:39 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=290] correct=1 wrong_fields=[]


In [16]:
evaluator_version="initial_test_onv3"
notegroupID=15
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("participants", pk) for pk in range(71, 73)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Notes Pepijn sessie 2 (application/vnd.google-apps.document)


2026-06-22 18:32:56 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=71:
{
  "full_name": "Sahar",
  "place_of_origin": "Syria",
  "municipality": "Venlo"
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1342 - Zorgcafé Venlo/2. Expertpool/Notes/Notes Pepijn sessie 2

--- Skipping PARTICIPANT: no URL ---

--- Evaluating participants pk=71 ---


2026-06-22 18:32:57 | INFO     | utils.token_logger | Token usage [participants pk=71] attempt 1 — in: 5139 (cached: 0), out: 33, cost: $0.006754
2026-06-22 18:32:57 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=71] correct=0 wrong_fields=['place_of_origin', 'municipality', 'session_identifier']
2026-06-22 18:32:57 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=72:
{
  "full_name": "Mehmet",
  "learning_route": "Z-route",
  "participant_group": "Asylum seeker",
  "place_of_origin": "Turkey",
  "language_group": [
    "Turkish"
  ],
  "municipality": "Venlo"
}



--- Evaluating participants pk=72 ---


2026-06-22 18:32:58 | INFO     | utils.token_logger | Token usage [participants pk=72] attempt 1 — in: 5164 (cached: 4096), out: 39, cost: $0.002237
2026-06-22 18:32:58 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=72] correct=0 wrong_fields=['learning_route', 'participant_group', 'place_of_origin', 'language_group', 'municipality']


In [17]:
evaluator_version="initial_test_onv3"
notegroupID=16
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("questions", 313)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Copy of iyad zorg cafe sessie 1.docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)


2026-06-22 18:38:53 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=313:
{
  "question_content": "Green/Red card statements\nReading out a statement and respond.\nI/people would face more problems with my/our physical and mental health.\nAccessing healthcare would be more difficult, especially for asylum seekers or those living in the AZC.\nI/people might feel isolated, unsupported, or unheard when dealing with health issues.\nWithout the Zorgcafé, I would not have anyone checking in on my mental health.\nI might have gone longer without medication or follow-up for my psychological issues.\nI would feel less safe and less supported in my daily life.\nBecause of the Zorgcafé:\nI/people can get the healthcare I/they need more easily.\nThe Zorgcafé provides guidance and support, including follow-up and medication management.\nI/people feel listened to and supported, which helps my/our mental health.\nIt serves as a bridge between me/people and oth

Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1342 - Zorgcafé Venlo/2. Expertpool/Notes/Copy of iyad zorg cafe sessie 1.docx

--- Skipping PARTICIPANT: no URL ---

--- Evaluating questions pk=313 ---


2026-06-22 18:38:54 | INFO     | utils.token_logger | Token usage [questions pk=313] attempt 1 — in: 4303 (cached: 0), out: 23, cost: $0.005609
2026-06-22 18:38:54 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=313] correct=1 wrong_fields=[]


In [18]:
evaluator_version="initial_test_onv3"
notegroupID=17
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("notegroups", 17)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Interview 3.18 BOOST (application/vnd.google-apps.document)


2026-06-22 18:40:50 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'notegroups' pk=17:
{
  "data_source_category": "diepteinterview"
}


Drive path: 01 extern/01 huidige klanten : projecten/1133 - AMIF Amsterdam/06 Diepteinterviews /Interviews BOOST/Interview 3.18 BOOST

--- Skipping PARTICIPANT: no URL ---

--- Evaluating notegroups pk=17 ---


2026-06-22 18:40:51 | INFO     | utils.token_logger | Token usage [notegroups pk=17] attempt 1 — in: 1521 (cached: 0), out: 23, cost: $0.002131
2026-06-22 18:40:51 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [notegroups pk=17] correct=1 wrong_fields=[]


In [19]:
evaluator_version="initial_test_onv3"
notegroupID=18
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("notegroups", 18)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Interview 3.21 (application/vnd.google-apps.document)


2026-06-22 18:42:26 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'notegroups' pk=18:
{
  "data_source_category": "diepteinterview"
}


Drive path: 01 extern/01 huidige klanten : projecten/1133 - AMIF Amsterdam/06 Diepteinterviews /Interviews BOOST/Interview 3.21

--- Skipping PARTICIPANT: no URL ---

--- Evaluating notegroups pk=18 ---


2026-06-22 18:42:29 | INFO     | utils.token_logger | Token usage [notegroups pk=18] attempt 1 — in: 1596 (cached: 0), out: 29, cost: $0.002285
2026-06-22 18:42:29 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [notegroups pk=18] correct=0 wrong_fields=['data_source_category', 'date']


In [20]:
evaluator_version="initial_test_onv3"
notegroupID=20
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("participants", pk) for pk in [86,88]]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: NOTITIES_IYAD (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/NOTITIES_IYAD

--- Loading and extracting text from PARTICIPANT ---
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)


2026-06-22 18:45:34 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=86:
{
  "full_name": "‪Reem Al abbas‬‏",
  "session_identifier": "R",
  "gender": "Vrouw / Woman / امرأة / Kadın",
  "learning_route": "B1-route",
  "place_of_origin": "Syrië",
  "language_group": [
    "Arabisch"
  ],
  "municipality": "MONNICKENDAM",
  "age": 19
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)

--- Evaluating participants pk=86 ---


2026-06-22 18:45:36 | INFO     | utils.token_logger | Token usage [participants pk=86] attempt 1 — in: 14820 (cached: 0), out: 29, cost: $0.018815
2026-06-22 18:45:36 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=86] correct=0 wrong_fields=['session_identifier', 'municipality']
2026-06-22 18:45:36 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=88:
{
  "full_name": "Riham almishael",
  "gender": "Vrouw / Woman / امرأة / Kadın",
  "learning_route": "Onderwijsroute",
  "place_of_origin": "سوريا",
  "language_group": [
    "عربي"
  ],
  "municipality": "MonnickendamCornelis Dirkszoonlaan 316",
  "age": 35
}



--- Evaluating participants pk=88 ---


2026-06-22 18:45:38 | INFO     | utils.token_logger | Token usage [participants pk=88] attempt 1 — in: 14817 (cached: 13696), out: 23, cost: $0.003343
2026-06-22 18:45:38 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=88] correct=1 wrong_fields=[]


In [21]:
evaluator_version="initial_test_onv3"
notegroupID=21
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("notegroups", 21)]
    +[("participants", pk) for pk in [91,92]]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: NOTITIES_Floris_en_Anne.docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/NOTITIES_Floris_en_Anne.docx

--- Loading and extracting text from PARTICIPANT ---
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)


2026-06-22 18:49:17 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'notegroups' pk=21:
{
  "data_source_category": "focus group"
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)

--- Evaluating notegroups pk=21 ---


2026-06-22 18:49:19 | INFO     | utils.token_logger | Token usage [notegroups pk=21] attempt 1 — in: 22421 (cached: 3712), out: 25, cost: $0.024100
2026-06-22 18:49:19 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [notegroups pk=21] correct=0 wrong_fields=['date']
2026-06-22 18:49:19 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=91:
{
  "full_name": "Musfira Mahnoor",
  "session_identifier": "Vrouw, 23, B1",
  "gender": "Vrouw / Woman / امرأة / Kadın",
  "learning_route": "Onderwijsroute",
  "participant_group": "Family migrant",
  "place_of_origin": "Pakistan",
  "language_group": [
    "English"
  ],
  "municipality": "Ilpendam",
  "age": 23
}



--- Evaluating participants pk=91 ---


2026-06-22 18:49:20 | INFO     | utils.token_logger | Token usage [participants pk=91] attempt 1 — in: 22955 (cached: 21888), out: 29, cost: $0.004360
2026-06-22 18:49:20 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=91] correct=0 wrong_fields=['participant_group', 'municipality']
2026-06-22 18:49:20 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'participants' pk=92:
{
  "full_name": "Angela Paola Barragán sanchez",
  "session_identifier": "Vrouw, 42, B1",
  "gender": "Vrouw / Woman / امرأة / Kadın",
  "learning_route": "B1-route",
  "participant_group": "Family migrant",
  "place_of_origin": "ColombiaC",
  "language_group": [
    "Español"
  ],
  "municipality": "Monnickendam",
  "age": 42
}



--- Evaluating participants pk=92 ---


2026-06-22 18:49:21 | INFO     | utils.token_logger | Token usage [participants pk=92] attempt 1 — in: 22961 (cached: 21888), out: 23, cost: $0.004307
2026-06-22 18:49:21 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [participants pk=92] correct=1 wrong_fields=[]


In [22]:
evaluator_version="initial_test_onv3"
notegroupID=22
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("questions", pk) for pk in range(391, 401)]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading: Notites_Mahad_2.docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/Notites_Mahad_2.docx

--- Loading and extracting text from PARTICIPANT ---
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)


2026-06-22 18:52:36 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=391:
{
  "question_content": "Hoe vind je het tempo en de sfeer in de lessen? Welke lessen heb je gevolgd?\nPast de manier van lesgeven bij jouw niveau en persoonlijke situatie?"
}


Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)

--- Evaluating questions pk=391 ---


2026-06-22 18:52:37 | INFO     | utils.token_logger | Token usage [questions pk=391] attempt 1 — in: 10472 (cached: 3712), out: 23, cost: $0.009144
2026-06-22 18:52:37 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=391] correct=1 wrong_fields=[]
2026-06-22 18:52:37 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=392:
{
  "question_content": "Wat zou je graag anders zien aan de inburgering?"
}



--- Evaluating questions pk=392 ---


...

2026-06-22 18:52:46 | INFO     | utils.token_logger | Token usage [questions pk=398] attempt 1 — in: 10447 (cached: 9344), out: 23, cost: $0.002777
2026-06-22 18:52:46 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=398] correct=1 wrong_fields=[]
2026-06-22 18:52:46 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=399:
{
  "question_content": "Door ons werk leren we vooral veel dagelijkse woordenschat."
}



--- Evaluating questions pk=399 ---


2026-06-22 18:52:47 | INFO     | utils.token_logger | Token usage [questions pk=399] attempt 1 — in: 10452 (cached: 9344), out: 29, cost: $0.002843
2026-06-22 18:52:47 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=399] correct=0 wrong_fields=['question_content', 'main_indicator']
2026-06-22 18:52:47 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'questions' pk=400:
{
  "question_content": "Voor ons horen werk en taallessen echt bij elkaar. Ze zijn niet hetzelfde, maar ze vullen elkaar aan."
}



--- Evaluating questions pk=400 ---


2026-06-22 18:52:48 | INFO     | utils.token_logger | Token usage [questions pk=400] attempt 1 — in: 10463 (cached: 0), out: 23, cost: $0.013309
2026-06-22 18:52:48 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [questions pk=400] correct=1 wrong_fields=[]


In [23]:
evaluator_version="initial_test_onv3"
notegroupID=23
text_doc, file_path_doc = textdoc_extractor(notegroupID, DB_PATH, service_account_file)
records_to_evaluate = (
    [("answers", pk) for pk in [1193,1208]]
)
for table_name, primary_key in records_to_evaluate:
    print(f"\n--- Evaluating {table_name} pk={primary_key} ---")
    try:
        evaluator = DB1recordEvaluator(
            db_path=DB_PATH,
            prompt_path_evaluator=prompt_path_evaluator,
            schema_path=schema_path,
            text_doc=text_doc,
            file_path_doc=file_path_doc,
            table_name=table_name,
            primary_key=primary_key,
            pipeline_type=pipeline_type,
            evaluator_version=evaluator_version,
        )
        evaluator.evalute_1json()
    except Exception as e:
        print(f"  FAILED: {e}")


--- Loading and extracting text from QA ---
Loading:  NOTITIES_Fatih (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/ NOTITIES_Fatih

--- Loading and extracting text from PARTICIPANT ---
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)


2026-06-22 18:55:54 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=1193:
{
  "questionID": 428,
  "participantID": 99,
  "answer_content_oriLAN": "Ik ben wiskundeleraar van beroep en mijn diploma is in Nederland erkend. Dat is fijn, maar de taal is nog steeds een grote uitdaging voor mij. Daarom doe ik nu mee aan een project voor statushouders in het onderwijs. In dit project volg ik een taalcursus en loop ik twee dagen per week stage op een school. Zo kan ik veel Nederlands oefenen in de praktijk.\nIk heb dit project gevonden via een Turkse vriendin die hier al eerder aan meedeed. Er is geen baan­garantie, maar ik krijg wel Nederlandse werkervaring. Dat vind ik belangrijk.\nWat soms spannend is: wiskundeleraar is geen tekortberoep meer. Dat betekent dat ik misschien ook moet nadenken over andere opties, een plan B of zelfs C. Toch blijf ik gemotiveerd, want ik wil graag in het onderwijs blijven werken en mezelf verder ontwikkelen.",
  "full_nam

Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)

--- Evaluating answers pk=1193 ---


2026-06-22 18:55:57 | INFO     | utils.token_logger | Token usage [answers pk=1193] attempt 1 — in: 16405 (cached: 0), out: 23, cost: $0.020736
2026-06-22 18:55:57 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=1193] correct=1 wrong_fields=[]
2026-06-22 18:55:57 | DEBUG    | oral_notes.evaluate.DB_1record_evaluator | DB2json result for 'answers' pk=1208:
{
  "questionID": 434,
  "participantID": 98,
  "answer_content_oriLAN": "Ja, tijdens mijn inburgering had ik soms stress. Dat kwam vooral door de taal. Ik was bang dat mijn Nederlands niet snel genoeg beter werd. Ook moest ik veel formulieren en brieven begrijpen. Soms wist ik niet precies wat ik moest doen. Dat gaf druk en zorgde ervoor dat ik veel nadacht over mijn toekomst.\nWat doet de gemeente volgens jou goed om stress te verminderen, en wat zou beter kunnen? (Bijvoorbeeld: duidelijkere informatie, meer persoonlijke uitleg, minder brieven tegelijk...)\nDe gemeente helpt goed door duidelijk te antw


--- Evaluating answers pk=1208 ---


2026-06-22 18:55:58 | INFO     | utils.token_logger | Token usage [answers pk=1208] attempt 1 — in: 16438 (cached: 15616), out: 23, cost: $0.003210
2026-06-22 18:55:58 | INFO     | oral_notes.evaluate.DB_1record_evaluator | Evaluation [answers pk=1208] correct=1 wrong_fields=[]
